<!-- nav -->
[← 3 · Links as rules](03_Links_As_Rules.ipynb) · [Home](../../README.md) · [5 · The shape of a population →](05_The_Shape_Of_A_Population.ipynb)

# 4 · Rules that take arguments

**The problem.** Many rules differ only in a value: "is named Ann", "is named Bob". Copying the rule once per value
copies its mistakes too. A predicate can take **parameters**, and be **applied** where it's used, by reference: the
same predicate object, written once.

In [1]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Constraints, Predicates, Queries, Validators
from mbse.Schemas.Framework import JSON
from toolkit import Contact, book, listed

store = book()
listed(store, *(store.Contact().name(n).age(a).create() for n, a in (("Ann", 34), ("Bob", 41), ("Kid", 9))))

## Step 1: a parameter

`.parameters(...)` takes property specs, as an object schema's `.properties(...)` does. The rule's free names are the
symbols and the parameters:

In [2]:
HasName = (Predicates.OfPredicate.Builder().name("HasName").symbols({"person": Contact})
           .parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())
list(HasName.symbols), list(HasName.parameters)

(['person'], ['name'])

## Step 2: applying it

Calling a predicate applies it: `HasName(who, "Ann")` binds its symbol, then its parameter. A builder gives the
variables it declares, so `named.c` is the variable `c`:

In [3]:
named = Predicates.OfPredicate.Builder().name("IsAnn").symbols({"c": Contact})
IsAnn = named.requires(HasName(named.c, "Ann")).create()
IsAnn.rule.predicate is HasName, [m["c"].name for m in Queries.Scan(store).select(IsAnn)]

(True, ['Ann'])

The application holds `HasName` itself, not a copy and not its name: change `HasName` and every rule that applies
it changes with it.

A query can give a predicate's parameters as variables; a validator refuses a predicate with parameters, since it holds
only where it's applied:

In [4]:
print([m["person"].name for m in Queries.Scan(store).select(HasName, {"name": "Bob"})])
try:
    Validators.Validate(store, [HasName])
except ValueError as error:
    print(error)

['Bob']
predicate 'HasName' has parameters: it is checked where it is applied


## Step 3: written once

A predicate applied in several places is one object, so a snapshot writes it once:

In [5]:
IsBob = Predicates.OfPredicate.Builder().name("IsBob").symbols({"c": Contact}).requires(HasName(E.variable("c"), "Bob")).create()
text = JSON.ToJSON(Constraints.Builders).Reachable(Constraints.OfSet.Schema, Constraints.check([IsAnn, IsBob]))
text.count('"name": "HasName"')

1

## Why it works this way

- **By reference, not by name.** A name is a promise that something with that name exists when the rule is read. A
  reference is the thing itself, written once and read back as one object.
- **Parameters, not copies.** A predicate with parameters is a rule with holes; applying it fills them. That keeps the
  rule in one place.

<!-- nav -->
[← 3 · Links as rules](03_Links_As_Rules.ipynb) · [Home](../../README.md) · [5 · The shape of a population →](05_The_Shape_Of_A_Population.ipynb)